# Credit Card Fraud Detection with Concept Drift (2026 Dataset)

## Overview
In addition to the anonymized European PCA dataset (`creditcard.csv`), this notebook trains and evaluates a second fraud detection pipeline on **real-world behavioral & banking features** (`credit_card_fraud_drift_2026.csv`).

Key goals:
1. Use **34 real-time, leakage-free banking features** (velocity, auth attempts, chargeback history, amount ratios).
2. Use a **temporal split** instead of a random split: Train on Phase 1 (Jan–Feb), validate on Phase 2 (Mar), and test across Phase 3 (Apr–May Peak Drift) and Phase 4 (Jun Stabilization).
3. Compare **Logistic Regression** vs **Random Forest** and save the best pipeline for the Streamlit app.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score, classification_report
)


## 1. Load Dataset & Inspect Drift Phases


In [ ]:
df = pd.read_csv('../data/credit_card_fraud_drift_2026.csv')
print('Shape:', df.shape)
print('Overall Fraud Rate:', f"{df['is_fraud'].mean()*100:.2f}%")
df.groupby(['drift_phase', 'drift_phase_name'])['is_fraud'].agg(['count', 'mean'])


## 2. Leakage-Free Feature Selection
We strictly exclude IDs, timestamps, drift metadata, and post-transaction/dispute columns (like `fraud_type`, `fraud_amount_lost`, `dispute_filed`, `pseudo_label`) so the model only uses features available at transaction time.


In [ ]:
FEATURES = [
    'transaction_amount', 'transaction_hour', 'card_present', 'customer_age',
    'account_age_days', 'credit_limit', 'credit_score', 'num_cards_held',
    'transactions_last_1h', 'transactions_last_24h', 'transactions_last_7d',
    'amount_last_1h', 'amount_last_24h', 'days_since_last_transaction',
    'distance_from_home_km', 'distance_from_last_tx_km', 'international_transaction',
    'high_risk_country', 'amount_vs_avg_ratio', 'used_chip', 'used_pin',
    'is_weekend', 'avg_transaction_amount_30d', 'std_transaction_amount_30d',
    'num_declined_last_30d', 'num_chargebacks_ever', 'failed_auth_attempts_24h',
    'device_change_7d', 'billing_shipping_mismatch', 'account_profile_change_7d',
    'merchant_velocity_24h', 'prior_dispute_ratio_90d', 'pos_entry_fallback',
    'pin_retry_count_24h'
]
TARGET = 'is_fraud'
print('Selected real-time features:', len(FEATURES))


## 3. Temporal Train / Validation / Drift-Test Split


In [ ]:
train_df = df[df['drift_phase'] == 1]  # Jan - Feb (Stable)
val_df   = df[df['drift_phase'] == 2]  # Mar (Pre-Drift Validation)
p3_df    = df[df['drift_phase'] == 3]  # Apr - May (Peak Drift)
p4_df    = df[df['drift_phase'] == 4]  # Jun (Partial Stabilization)

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_val, y_val     = val_df[FEATURES], val_df[TARGET]
print(f'Train: {X_train.shape} | Val: {X_val.shape} | Phase 3: {p3_df.shape} | Phase 4: {p4_df.shape}')


## 4. Train & Compare Models


In [ ]:
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

rf_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(
        n_estimators=100, max_depth=12, min_samples_leaf=5,
        class_weight='balanced', random_state=42
    ))
])

for name, pipe in [('Logistic Regression', lr_pipe), ('Random Forest', rf_pipe)]:
    pipe.fit(X_train, y_train)
    probs = pipe.predict_proba(X_val)[:, 1]
    preds = (probs >= 0.5).astype(int)
    print(f'{name:20s} | ROC-AUC: {roc_auc_score(y_val, probs):.4f} | PR-AUC: {average_precision_score(y_val, probs):.4f} | F1: {f1_score(y_val, preds):.4f}')


## 5. Evaluate Random Forest Across All Drift Phases


In [ ]:
results = []
for label, sub in [
    ('Phase 2: Pre-Drift (Mar)', val_df),
    ('Phase 3: Peak Drift (Apr-May)', p3_df),
    ('Phase 4: Stabilization (Jun)', p4_df)
]:
    probs = rf_pipe.predict_proba(sub[FEATURES])[:, 1]
    preds = (probs >= 0.5).astype(int)
    results.append({
        'Phase': label,
        'ROC-AUC': round(roc_auc_score(sub[TARGET], probs), 4),
        'PR-AUC': round(average_precision_score(sub[TARGET], probs), 4),
        'Precision': round(precision_score(sub[TARGET], preds), 4),
        'Recall': round(recall_score(sub[TARGET], preds), 4),
        'F1': round(f1_score(sub[TARGET], preds), 4),
    })

pd.DataFrame(results)


## 6. Top Driving Banking Features & Save Pipeline


In [ ]:
importances = pd.Series(rf_pipe.named_steps['model'].feature_importances_, index=FEATURES)
importances.sort_values(ascending=False).head(10).plot(kind='barh', figsize=(8, 4), title='Top 10 Fraud Signals')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

joblib.dump(rf_pipe, '../model/drift_fraud_pipeline.joblib', compress=3)
print('Saved ../model/drift_fraud_pipeline.joblib')
